<a href="https://colab.research.google.com/github/christopher-alford/mlb-nlp-platform/blob/main/4050_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas numpy

In [ ]:
import pandas as pd

mlb = pd.read_csv("oddsDataMLB.csv")
mlb['date'] = pd.to_datetime(mlb['date'])

def make_game_id(row):
    teams = "_".join(sorted([row['team'], row['opponent']]))
    return f"{row['date'].date()}_{teams}"

mlb['GameID'] = mlb.apply(make_game_id, axis=1)
mlb['win'] = (mlb['runs'] > mlb['oppRuns']).astype(int)
mlb['run_diff'] = mlb['runs'] - mlb['oppRuns']


In [ ]:
# Remove duplicate columns by keeping the first occurrence
mlb = mlb.loc[:, ~mlb.columns.duplicated()]


In [ ]:
import numpy as np

def american_to_prob(odds):
    if pd.isna(odds):
        return np.nan
    if odds > 0:
        return 100 / (odds + 100)
    else:
        return -odds / (-odds + 100)

mlb['prob_team'] = mlb['moneyLine'].apply(american_to_prob)
mlb['prob_opp'] = mlb['oppMoneyLine'].apply(american_to_prob)


In [ ]:
mlb['prob_total'] = mlb['prob_team'] + mlb['prob_opp']
mlb['prob_team_norm'] = mlb['prob_team'] / mlb['prob_total']
mlb['prob_opp_norm'] = mlb['prob_opp'] / mlb['prob_total']


In [ ]:
def summarize_features(row):
    # --- DATE ---
    # Ensure row['date'] is a datetime
    date_val = pd.to_datetime(row['date'], errors='coerce')

    if pd.isna(date_val):
        date_str = "Unknown Date"
    else:
        date_str = date_val.strftime("%B %d, %Y")

    #STATS
    team = row['team']
    opp = row['opponent']
    runs = row['runs']
    opp_runs = row['oppRuns']
    diff = row['run_diff']

    #FAVORITE
    favored = "were favored" if row.get('prob_team_norm', 0.5) > 0.5 else "were underdogs"

    #UPSET
    upset = ""
    if row['win'] == 1 and row.get('prob_team_norm', 0.5) < 0.5:
        upset = "in an upset victory "
    if row['win'] == 0 and row.get('prob_team_norm', 0.5) > 0.5:
        upset = "in a surprising loss "

    #OUTCOME
    outcome_sentence = (
        f"{team} defeated {opp} {runs} – {opp_runs}"
        if row['win'] == 1 else
        f"{team} lost to {opp} {opp_runs} – {runs}"
    )

    return {
        "date_str": date_str,
        "team": team,
        "opponent": opp,
        "runs": runs,
        "opp_runs": opp_runs,
        "diff": diff,
        "favored": favored,
        "upset_phrase": upset,
        "outcome_sentence": outcome_sentence
    }



In [ ]:
for col in ["date_str", "favored", "upset_phrase", "diff", "outcome_sentence"]:
    if col in mlb.columns:
        del mlb[col]


In [ ]:
def generate_summary(row):
    base = f"On {row['date_str']}, {row['outcome_sentence']}."

    odds_line = f" They {row['favored']} entering the game."

    if row['upset_phrase']:
        odds_line = " This came " + row['upset_phrase'] + "despite the odds."

    diff_line = (
        f" The run differential was {row['diff']}."
        if row['diff'] != 0 else
        " The game was decided by a single run."
    )

    return base + odds_line + diff_line



In [ ]:

mlb['date'] = pd.to_datetime(mlb['date'], errors='coerce')


cols_to_delete = [c for c in mlb.columns if c.lower() in ['date_str', 'datestr', 'formatted_date']]
for c in cols_to_delete:
    del mlb[c]



In [ ]:
mlb_features = mlb.apply(summarize_features, axis=1, result_type="expand")


In [ ]:
print(mlb_features.head())


         date_str team opponent  runs  opp_runs  diff         favored  \
0  March 28, 2012  SEA      OAK     3         1     2    were favored   
1  March 28, 2012  OAK      SEA     1         3    -2  were underdogs   
2  March 29, 2012  SEA      OAK     1         4    -3  were underdogs   
3  March 29, 2012  OAK      SEA     4         1     3    were favored   
4  April 04, 2012  STL      MIA     4         1     3  were underdogs   

           upset_phrase        outcome_sentence  
0                        SEA defeated OAK 3 – 1  
1                         OAK lost to SEA 3 – 1  
2                         SEA lost to OAK 4 – 1  
3                        OAK defeated SEA 4 – 1  
4  in an upset victory   STL defeated MIA 4 – 1  


In [ ]:
mlb = pd.concat([mlb, mlb_features], axis=1)


In [ ]:
"group" in mlb.columns
"date_str" in mlb.columns


True

In [ ]:
mlb["summary"] = mlb.apply(generate_summary, axis=1)

In [ ]:
mlb.to_csv("mlb_game_summaries.csv", index=False)

In [ ]:
mlb[["date_str","team","opponent","summary"]].sample(5)


,date_str,team,team,opponent,opponent,summary
31927,"July 11, 2018",BOS,BOS,TEX,TEX,"On July 11, 2018, BOS defeated TEX 4 – 2. They..."
121,"April 10, 2012",HOU,HOU,ATL,ATL,"On April 10, 2012, HOU lost to ATL 6 – 4. They..."
19427,"October 04, 2015",BAL,BAL,NYY,NYY,"On October 04, 2015, BAL defeated NYY 9 – 4. T..."
41200,"April 21, 2021",BOS,BOS,TOR,TOR,"On April 21, 2021, BOS lost to TOR 6 – 3. This..."
630,"April 29, 2012",LAA,LAA,CLE,CLE,"On April 29, 2012, LAA lost to CLE 4 – 0. This..."


In [ ]:
from google.colab import files
files.download("mlb_game_summaries.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install scikit-learn

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [ ]:
features = [
    "prob_team_norm",
    "prob_opp_norm",
    "moneyLine",
    "oppMoneyLine"
]

X = mlb[features]
y = mlb["win"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)


LogisticRegression(max_iter=500)

In [ ]:
preds = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, preds))


Accuracy: 0.5825829123654733


In [ ]:
!pip install joblib

In [ ]:
import joblib
joblib.dump(model, "mlb_model.pkl")


['mlb_model.pkl']